<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

### **0. Konfigurasi parameter eksperimen**

In [10]:
# Nama file dataset
DATASET = "pci-ml-26.xlsx"

# Parameter split data
SPLIT_PARAMS = {
    "n_seed": 20000,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

# Cross Validation
CV_SPLITS = 5

# Parameter tuning Random Forest
RF_N_ITER = 100

# Parameter tuning XGBoost


# Parameter tuning ANN


# Model seed (RF, XGBoost, ANN)
SEED = 42

# Import
from pathlib import Path
from IPython.display import display, HTML, Markdown
from pci_ml.memory import release

### **1. Pembersihan data dan rekayasa fitur**

In [11]:
# Memuat dataset, membersihkan data, dan menambahkan fitur agregat
from pci_ml.data import load_data, clean_data, to_density, engineer_features
from pci_ml.ui import df_preview

# muat dataset mentah
raw = load_data(DATASET)
# membersihkan data duplikat, data tidak valid
df, features = clean_data(raw)
# menghitung fitur density
df, features = to_density(df, features)
# feature engineering: total densitas, total densitas per tingkat keparahan (L/M/H)
df, features = engineer_features(df, features)

print(f"File dimuat: {DATASET} | Data telah dibersihkan | {len(features)} fitur telah dihitung.")
display(HTML(df_preview(df)))

release("raw")

File dimuat: pci-ml-26.xlsx | Data telah dibersihkan | 59 fitur telah dihitung.


ROAD_NAME,ROAD_ID,SEGMENT_ID,GATOR_CRACK_L,GATOR_CRACK_M,GATOR_CRACK_H,…,PCI
Guru Bangkol,R001,S001,0,0,0,…,85.5
Guru Bangkol,R001,S002,0,0.0039,0,…,73.34
Guru Bangkol,R001,S003,0,0,0,…,89.94
Guru Bangkol,R001,S004,0.0014,0.0231,0,…,73.63
Guru Bangkol,R001,S005,0,0,0,…,96.19
Guru Bangkol,R001,S006,0,0,0,…,84.01
Guru Bangkol,R001,S007,0,0,0,…,90.7
Guru Bangkol,R001,S008,0.0243,0,0,…,73.72
Guru Bangkol,R001,S009,0,0,0.0044,…,77.9
Guru Bangkol,R001,S010,0,0,0,…,84.22


### **2. Persiapan input model *machine learning***

##### Pemisahan data latih dan data uji

In [12]:
# Memisah data sesuai kriteria:
from pci_ml.data import DataSplit
from pci_ml.memory import release

print(f"Jumlah seed dicari: {SPLIT_PARAMS["n_seed"]} | "
      f"Test size: {SPLIT_PARAMS["test_size"]} | "
      f"Toleransi distribusi: {SPLIT_PARAMS["share_tol"]} | "
      f"PCI minimum: {SPLIT_PARAMS["pci_min"]}")

# Kriteria 1: test, train = 80%, 20% (dengan toleransi 1%)
# Kriteria 2: tidak ada fitur yang hanya muncul di test
# Kriteria 3: nilai target ekstrim, test harus punya jalan dengan PCI = 0
result = DataSplit(df, features, DATASET, SPLIT_PARAMS).run()

print(result["summary"])

release("result", "df", "features")

Jumlah seed dicari: 20000 | Test size: 0.2 | Toleransi distribusi: 0.01 | PCI minimum: 0
Seed used: 16650 (Gap: 0.2172)
Train: 935 rows (79.44%) | Test: 242 rows (20.56%)
Roads: 27 train | 7 test
Saved to: "C:\Users\gearb\Desktop\pci-ml\datasets\pci-ml-26\n20000_t0.2_s0.01_p0"


##### Muat data latih dan data uji

In [13]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import ModelInput, PCI, ROAD_ID, METADATA
from pci_ml.model import make_folds

# Muat file dataset yang sudah dipisah (train/test, features, dan metadata)
mi = ModelInput(DATASET)
tag = mi.find(SPLIT_PARAMS)
train_df, test_df, features, meta = mi.load(tag)

# definisi data untuk kebutuhan input model
train_features  = train_df[features].to_numpy(dtype="float32")
test_features   = test_df[features].to_numpy(dtype="float32")
train_pci       = train_df[PCI].to_numpy(dtype="float32") # PCI scale 0-100; divide by 100 in the ANN cell
test_pci        = test_df[PCI].to_numpy(dtype="float32")
train_road_ids  = train_df[ROAD_ID].to_numpy()
test_road_info  = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)

# buat fold untuk k-fold cross validation
CV_FOLDS = make_folds(train_road_ids, CV_SPLITS)

# Summary
n_train, n_test = len(train_pci), len(test_pci)
total = n_train + n_test
split_text = (f"**Train:** {train_features.shape} ({n_train / total:.2%}) | "
              f"**Test:** {test_features.shape} ({n_test / total:.2%})")
display(Markdown(f"**Split terpilih:** `{tag}`\n\n{split_text}"))

release("mi", "tag", "train_df", "test_df", "meta", "n_train", "n_test", "total", "split_text")

**Split terpilih:** `n20000_t0.2_s0.01_p0`

**Train:** (935, 59) (79.44%) | **Test:** (242, 59) (20.56%)

### **3. Pelatihan dan pengujian model *Random Forest***

##### Hyperparameter tuning dan pelatihan model *random forest*

In [14]:
# Tuning hyperparameter dan pelatihan model Random Forest final
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterSampler

from pci_ml.model import score_prediction
from pci_ml.random_forest import best_rf_candidate, save_rf, load_rf, show_rf_report
from pci_ml.ui import build_hyperparameter_table, format_table

print(f"Tuning Random Forest: {RF_N_ITER} iterasi | {CV_SPLITS} fold CV")

# Hyperparameter tuning untuk model random forest
RF_SEARCH_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3, 0.4],
    "max_samples": [0.5, 0.6, 0.7, 0.8, 0.9],
}
rf_candidates = ParameterSampler(RF_SEARCH_SPACE, n_iter=RF_N_ITER, random_state=SEED)
rf_best_mae, rf_best_params, rf_oof = best_rf_candidate(rf_candidates, train_features, train_pci, SEED, CV_FOLDS)
release("rf_candidates")

# Training model dengan hyperparameter terbaik
tuned_forest = RandomForestRegressor(**rf_best_params, random_state=SEED, n_jobs=-1)
tuned_forest.fit(train_features, train_pci)

# Save model random forest
save_rf(tuned_forest, DATASET, SPLIT_PARAMS)
release ("tuned_forest")

Tuning Random Forest: 100 iterasi | 5 fold CV
Model saved to models/pci-ml-26_n20000_t0.2_s0.01_p0/pci-ml-26_rf.joblib


##### Muat model *random forest* dan prediksi pada data uji

In [16]:
# Load model random forest
tuned_forest = load_rf(DATASET, SPLIT_PARAMS)

# k-fold cv score for random forest
rf_cv_score = score_prediction(train_pci, rf_oof)

# prediksi model terhadap data test
rf_pci_predict = tuned_forest.predict(test_features)
rf_test_score = score_prediction(test_pci, rf_pci_predict)

show_rf_report(rf_best_params, RF_SEARCH_SPACE, rf_cv_score, rf_test_score)

##### **Hyperparameter terbaik**

Hyperparameter,Nilai,Keterangan
n_estimators,300,Jumlah pohon keputusan.
max_depth,20,Kedalaman maksimum tiap pohon.
min_samples_split,4,Jumlah sampel minimum agar sebuah node boleh dipecah.
min_samples_leaf,1,Jumlah sampel minimum di tiap daun.
max_features,0.3,Fraksi fitur yang dipertimbangkan di tiap pohon.
max_samples,0.9,Fraksi data latih yang dipakai untuk tiap pohon.


##### **Hasil evaluasi**

Evaluasi,MAE,RMSE,R²
Cross-Validation,5.8378,8.8181,0.8502
Test,5.7101,8.8617,0.8465
